# PB01 — BCI Illiteracy Prediction from Rest EEG

**Obiettivo**: validare la claim di Blankertz et al. — le caratteristiche EEG a riposo predicono la performance nel task di imagined speech.

**Pipeline**:
1. Carica epoche `riposo_NNN.csv` per ogni soggetto
2. Estrae feature spettrali (alpha, beta, SMR power) e di connettività
3. Correla con la `bacc` soggetto-specifica (da W&B / EEG_37)
4. Classificatore binario: BCI-literate vs BCI-illiterate

**Riferimento**: Blankertz et al. 2010 — *Neurophysiological predictor of SMR-based BCI performance*

In [ ]:
# ============================================================
# CONFIG
# ============================================================
from pathlib import Path

project_root = next(
    (p for p in [Path().resolve()] + list(Path().resolve().parents) if (p / '.git').exists()),
    Path().resolve()
)

DATA_ROOT = project_root / 'data' / 'raw_csv' / 'training_set'

SFREQ   = 256
N_CHAN  = 61
N_SAMP  = 384

# Soglia per definire BCI-illiterate (bacc <= threshold = illiterate)
ILLITERACY_THRESHOLD = 0.30  # 30% bacc su 4 classi (chance=25%)

print(f'DATA_ROOT: {DATA_ROOT}')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal, stats
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneOut
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print('Import OK')

In [ ]:
# ============================================================
# CARICA BACC PER SOGGETTO
# Fonte: PB00_compute_subject_bacc.ipynb (CSV locale)
#        oppure W&B se preferisci usare EEG_06 / EEG_13b
# ============================================================

BACC_CSV = project_root / 'data' / 'interim' / 'subject_bacc_pipelineB.csv'

if BACC_CSV.exists():
    _df_bacc = pd.read_csv(BACC_CSV).dropna(subset=['bacc'])
    SUBJ_BACC = dict(zip(_df_bacc['subj_id'].astype(int), _df_bacc['bacc']))
    print(f'Caricati {len(SUBJ_BACC)} soggetti da {BACC_CSV.name}')
    print(f'bacc media: {_df_bacc["bacc"].mean():.3f}  range: [{_df_bacc["bacc"].min():.3f}, {_df_bacc["bacc"].max():.3f}]')
else:
    print('WARNING: subject_bacc_pipelineB.csv non trovato.')
    print('Esegui PB00_compute_subject_bacc.ipynb prima di questo notebook.')
    SUBJ_BACC = {}  # notebook continua senza correlazioni

In [ ]:
# ============================================================
# CARICA EPOCHE RIPOSO PER SOGGETTO
# Media su sessioni → (n_riposo_tot, 61, 384) per soggetto
# ============================================================

def load_rest_epochs(subj_id: int, data_root: Path) -> np.ndarray:
    """Carica tutte le epoche riposo di un soggetto (tutte le sessioni).
    Ritorna array (n_epochs, 61, 384) float32."""
    epochs = []
    for sess_dir in sorted(data_root.glob(f'P{subj_id:03d}_S*')):
        for f in sorted(sess_dir.glob('riposo_*.csv')):
            x = pd.read_csv(f, header=None).values.astype(np.float32)
            if x.shape == (N_CHAN, N_SAMP):
                epochs.append(x)
    return np.array(epochs) if epochs else np.empty((0, N_CHAN, N_SAMP), dtype=np.float32)


# Test su un soggetto
test_rest = load_rest_epochs(0, DATA_ROOT)
print(f'P000: {test_rest.shape} epoche riposo')  # atteso: (n, 61, 384)

In [ ]:
# ============================================================
# ESTRAZIONE FEATURE SPETTRALI (Blankertz-style)
# ============================================================

BANDS = {
    'delta': (1, 4),
    'theta': (4, 8),
    'alpha': (8, 12),
    'SMR':   (12, 15),   # sensorimotor rhythm — chiave in Blankertz
    'beta':  (15, 30),
    'gamma': (30, 45),
}

def bandpower(epochs: np.ndarray, band: tuple, sfreq: int = SFREQ) -> np.ndarray:
    """Potenza media nella banda per ogni canale.
    Input: (n_epochs, n_chan, n_samp)
    Output: (n_chan,) — media su epoche"""
    freqs, psd = signal.welch(epochs, fs=sfreq, nperseg=SFREQ, axis=-1)  # (n, ch, freqs)
    idx = np.logical_and(freqs >= band[0], freqs <= band[1])
    return psd[:, :, idx].mean(axis=(0, 2))  # (n_chan,)


def extract_rest_features(epochs: np.ndarray) -> np.ndarray:
    """Feature vector per soggetto dal rest.
    Ritorna vettore 1D: bandpower per ogni banda × canale."""
    if len(epochs) == 0:
        return None
    feats = []
    for band_name, band in BANDS.items():
        bp = bandpower(epochs, band)  # (n_chan,)
        feats.append(np.log1p(bp))   # log-transform per stabilizzare
    return np.concatenate(feats)     # (n_bands * n_chan,)


# Test
feat_test = extract_rest_features(test_rest)
print(f'Feature vector shape: {feat_test.shape}')  # atteso: (6*61=366,)

In [ ]:
# ============================================================
# ESTRAZIONE FEATURE PER TUTTI I SOGGETTI
# ============================================================

all_subj = sorted(set(
    int(d.name.split('_')[0][1:]) for d in DATA_ROOT.iterdir() if d.is_dir()
))
print(f'Soggetti trovati: {len(all_subj)}')

features, labels, subj_ids = [], [], []

for sid in tqdm(all_subj, desc='Soggetti'):
    epochs = load_rest_epochs(sid, DATA_ROOT)
    if len(epochs) == 0:
        continue
    feat = extract_rest_features(epochs)
    if feat is None:
        continue
    features.append(feat)
    subj_ids.append(sid)
    # Label BCI literacy (solo se abbiamo bacc)
    if sid in SUBJ_BACC:
        labels.append(1 if SUBJ_BACC[sid] > ILLITERACY_THRESHOLD else 0)
    else:
        labels.append(np.nan)

X = np.array(features)          # (n_subj, n_features)
y_bacc = np.array([SUBJ_BACC.get(s, np.nan) for s in subj_ids])
y_lit  = np.array(labels)

print(f'X: {X.shape} | soggetti con bacc: {(~np.isnan(y_bacc)).sum()}')

In [ ]:
# ============================================================
# CORRELAZIONE UNIVARIATA REST ↔ BACC (continua)
#   Spearman per feature + correzione multipla (BH-FDR) + null per
#   permutazione sul max|ρ| (family-wise, controlla i ~366 test in blocco)
# ============================================================
from scipy.stats import rankdata

mask = ~np.isnan(y_bacc)
X_lab = X[mask]
y_lab = y_bacc[mask]
n_subj, n_feat = X_lab.shape
print(f'Soggetti con bacc: {n_subj} | feature: {n_feat}')

def _zcols(A):
    A = A - A.mean(0); s = A.std(0); s[s == 0] = 1.0
    return A / s

# Spearman = Pearson sui ranghi → vettorizzato (serve per la permutazione)
RX = _zcols(np.apply_along_axis(rankdata, 0, X_lab))        # (n, p) ranghi z-scored
ry = _zcols(rankdata(y_lab).reshape(-1, 1)).ravel()         # (n,)
rhos = (RX.T @ ry) / n_subj                                 # (p,) Spearman ρ
# p-value univariato two-sided (approx t)
t = rhos * np.sqrt((n_subj - 2) / np.clip(1 - rhos**2, 1e-12, None))
pvals = 2 * stats.t.sf(np.abs(t), n_subj - 2)

# Benjamini–Hochberg FDR (manuale, niente dipendenze extra)
def bh_fdr(p):
    p = np.asarray(p, float); n = len(p); order = np.argsort(p)
    q = p[order] * n / (np.arange(n) + 1)
    q = np.minimum.accumulate(q[::-1])[::-1]
    out = np.empty(n); out[order] = np.clip(q, 0, 1)
    return out
p_fdr = bh_fdr(pvals)
n_sig = int((p_fdr < 0.05).sum())

# Null per permutazione sul max|ρ| (family-wise su tutte le feature)
rng = np.random.default_rng(42); N_PERM = 2000
max_null = np.empty(N_PERM)
for k in range(N_PERM):
    rp = (RX.T @ rng.permutation(ry)) / n_subj
    max_null[k] = np.abs(rp).max()
obs_max = np.abs(rhos).max()
p_fw = (np.sum(max_null >= obs_max) + 1) / (N_PERM + 1)

print(f'Feature significative dopo FDR (q<0.05): {n_sig} / {n_feat}')
print(f'Max|ρ| osservato = {obs_max:.3f}  |  p family-wise (permutazione) = {p_fw:.4f}')
print('→ se p_fw non è <0.05, nessuna singola feature di riposo correla con la bacc oltre il caso.\n')

bn = list(BANDS.keys())
print('Top 15 feature per |ρ| (ρ, p, q_FDR):')
for i in np.argsort(np.abs(rhos))[::-1][:15]:
    print(f'  [{bn[i // N_CHAN]:5s} ch{i % N_CHAN:02d}]  ρ={rhos[i]:+.3f}  p={pvals[i]:.4f}  q={p_fdr[i]:.3f}')

In [ ]:
# ============================================================
# (secondario) CLASSIFICATORE BINARIO literate vs illiterate — LOO-CV
#   ATTENZIONE: con bAcc ~chance la soglia rende i gruppi MOLTO sbilanciati,
#   quindi l'AUC va letta contro un null per permutazione. Il risultato
#   principale resta il predittore CONTINUO della cella sopra.
# ============================================================
lit_mask = mask & ~np.isnan(y_lit)
X_cls = X[lit_mask]
y_cls = y_lit[lit_mask].astype(int)
print(f'Literate (bacc>{ILLITERACY_THRESHOLD}): {y_cls.sum()}  |  Illiterate: {(y_cls==0).sum()}  '
      f'→ sbilanciamento {y_cls.mean():.0%}')

if y_cls.sum() < 3 or (y_cls == 0).sum() < 3:
    print('Gruppi troppo sbilanciati per un classificatore affidabile — salto (usa il predittore continuo).')
else:
    def loo_auc(Xc, yc):
        yt, yp = [], []
        for tr, te in LeaveOneOut().split(Xc):
            sc = StandardScaler().fit(Xc[tr])
            clf = LogisticRegression(C=0.1, max_iter=1000).fit(sc.transform(Xc[tr]), yc[tr])
            yt.append(yc[te][0]); yp.append(clf.predict_proba(sc.transform(Xc[te]))[0, 1])
        return roc_auc_score(yt, yp)

    auc = loo_auc(X_cls, y_cls)
    rng = np.random.default_rng(1); N_PERM_AUC = 1000
    null_auc = np.array([loo_auc(X_cls, rng.permutation(y_cls)) for _ in range(N_PERM_AUC)])
    p_auc = (np.sum(null_auc >= auc) + 1) / (N_PERM_AUC + 1)
    print(f'LOO-CV AUC = {auc:.3f}  |  null permutazione media={null_auc.mean():.3f}  p={p_auc:.4f}')

In [ ]:
# ============================================================
# CLASSIFICATORE BCI-LITERATE vs BCI-ILLITERATE (LOO-CV)
# ============================================================

lit_mask = mask & ~np.isnan(y_lit)
X_cls = X[lit_mask]
y_cls = y_lit[lit_mask].astype(int)

print(f'Literate: {y_cls.sum()}  Illiterate: {(y_cls==0).sum()}')

loo  = LeaveOneOut()
scaler = StandardScaler()
clf    = LogisticRegression(C=0.1, max_iter=1000)

y_true_all, y_prob_all = [], []
for train_idx, test_idx in loo.split(X_cls):
    X_tr = scaler.fit_transform(X_cls[train_idx])
    X_te = scaler.transform(X_cls[test_idx])
    clf.fit(X_tr, y_cls[train_idx])
    y_true_all.append(y_cls[test_idx][0])
    y_prob_all.append(clf.predict_proba(X_te)[0, 1])

auc = roc_auc_score(y_true_all, y_prob_all)
print(f'LOO-CV AUC: {auc:.3f}')

In [ ]:
# ============================================================
# PLOT: alpha/SMR power vs bacc (scatter per soggetto)
# ============================================================

# Media alpha power su elettrodi centrali (Cz, C3, C4 ≈ indici 25,24,26)
alpha_idx = 2  # banda alpha = indice 2 in BANDS
central_chs = [24, 25, 26]  # C3, Cz, C4 approssimati
alpha_central = X_labeled[:, alpha_idx * N_CHAN: (alpha_idx+1) * N_CHAN][:, central_chs].mean(axis=1)

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(alpha_central, y_labeled, alpha=0.7, edgecolors='k', linewidths=0.5)
rho, p = stats.spearmanr(alpha_central, y_labeled)
ax.set_xlabel('log Alpha power (C3/Cz/C4) — rest')
ax.set_ylabel('bacc imagined speech')
ax.set_title(f'Rest alpha vs BCI performance  ρ={rho:.3f}  p={p:.4f}')
ax.axhline(ILLITERACY_THRESHOLD, color='r', linestyle='--', label='illiteracy threshold')
ax.legend()
plt.tight_layout()
plt.savefig(project_root / 'figures' / 'PB01_rest_alpha_vs_bacc.png', dpi=150)
plt.show()